# Decision Tree
---

## What Is a Decision Tree?

A **Decision Tree** is a **supervised learning algorithm** used for **classification and regression** that splits data into branches based on feature conditions.

---

## Core Idea

Recursively split the dataset so that each split **reduces impurity (classification)** or **error (regression)**.

---

## Tree Components

| Component | Meaning |
|---------|--------|
| Root Node | First and most important split |
| Decision Node | Condition on a feature |
| Leaf Node | Final prediction |
| Branch | Outcome of a condition |

---

## How Splitting Works

At each node:
1. Evaluate all features
2. Choose the best split
3. Partition the data
4. Repeat recursively

---

## Split Criteria

### Classification
- **Gini Impurity**
- **Entropy (Information Gain)**

### Regression
- **Mean Squared Error (MSE)**
- **Mean Absolute Error (MAE)**

---

## Stopping Conditions

- Node becomes pure
- Maximum depth reached
- Minimum samples per node reached

---

## Overfitting Control

- max_depth
- min_samples_split
- min_samples_leaf
- Pruning

---

## One-line Definition

**A Decision Tree makes predictions by recursively splitting data based on feature conditions to minimize impurity or error.**

---


In [0]:
import numpy as np
import pandas as pd

In [0]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


In [0]:
class DecisionTreeRegressor:
    """
    Decision Tree Regressor built from scratch.
    """

    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf = 1, max_features = None):
        """
        Initialize the decision tree with hyperparameters.

        Parameters:
        max_depth : int
            Maximum depth of the tree
        min_samples_split : int
            Minimum number of samples required to split a node
        """

        # Hyperparameters
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features



        # Root node of the tree (will be set after training)
        self.root = None

    def fit(self, X, y):
        """
        Train the decision tree regressor.

        Parameters:
        X : pandas DataFrame (m x n)
            Feature matrix
        y : pandas Series or DataFrame (m,)
            Target values
        """

        # Convert Pandas inputs to NumPy arrays
        X = X.values
        y = y.values

        # Validate input shapes
        if X.shape[0] != y.shape[0]:
            raise ValueError("Number of samples in X and y must be the same")

        # Start building the tree from root
        self.root = self._build_tree(X, y, depth=0)

    def _build_tree(self, X, y, depth):
        """
        Recursively build the decision tree.

        Parameters:
        X : numpy array (m x n)
            Feature matrix for current node
        y : numpy array (m,)
            Target values for current node
        depth : int
            Current depth of the tree

        Returns:
        node : Node
            A decision node or a leaf node
        """

        # Step 1: Check stopping conditions
        if self._check_stopping_conditions(y, depth):
            # Create and return a leaf node
            return self._create_leaf_node(y)

        # Step 2: Find the best split
        feature, threshold, X_left, y_left, X_right, y_right = self._find_best_split(X, y)

        # Step 3: If no valid split is found, create a leaf
        if X_left is None or X_right is None:
            return self._create_leaf_node(y)

        # Step 4: Create a decision node
        node = self._create_decision_node(feature, threshold)

        # Step 5: Recursively build left and right subtrees
        node.left = self._build_tree(X_left, y_left, depth + 1)
        node.right = self._build_tree(X_right, y_right, depth + 1)

        # Step 6: Return the node
        return node


    def _check_stopping_conditions(self, y, depth):
        """
        Determines whether the decision tree should stop splitting
        and create a leaf node.

        Parameters:
        y : numpy array (m,)
            Target values at current node
        depth : int
            Current depth of the tree

        Returns:
        bool
            True  -> stop splitting
            False -> continue splitting
        """

        # Stop if maximum depth is reached
        if self.max_depth is not None and depth >= self.max_depth:
            return True

        # Stop if not enough samples to split further
        if len(y) < self.min_samples_split:
            return True

        # Stop if target values have zero variance (pure node)
        if y.var() == 0:
            return True

        # Otherwise, continue splitting
        return False

    def _find_best_split(self, X, y):
        """
        Find the best feature and threshold to split the data.
        """

        best_error = float("inf")
        best_feature = None
        best_threshold = None
        best_X_left = best_y_left = None
        best_X_right = best_y_right = None

        n_features = X.shape[1]
        
        k = self._get_max_features(n_features)
        features = np.random.choice(
            n_features,
            size=k,
            replace=False
        )
        for feature_index in features:

            feature_values = X[:, feature_index]
            thresholds = np.unique(feature_values)

            for threshold in thresholds:

                left_mask = feature_values <= threshold
                right_mask = feature_values > threshold

                # enforce min_samples_leaf
                if (
                    left_mask.sum() < self.min_samples_leaf or
                    right_mask.sum() < self.min_samples_leaf
                ):
                    continue

                X_left = X[left_mask]
                y_left = y[left_mask]
                X_right = X[right_mask]
                y_right = y[right_mask]

                error = self._weighted_mean_squared_error(y_left, y_right)

                if error < best_error:
                    best_error = error
                    best_feature = feature_index
                    best_threshold = threshold
                    best_X_left = X_left
                    best_y_left = y_left
                    best_X_right = X_right
                    best_y_right = y_right

        if best_feature is None:
            return None, None, None, None, None, None

        return (
            best_feature,
            best_threshold,
            best_X_left,
            best_y_left,
            best_X_right,
            best_y_right
        )


    def _create_leaf_node(self, y):
        """
        Create a leaf node for regression.

        Parameters:
        y : numpy array (m,)
            Target values at the leaf

        Returns:
        Node
            Leaf node with prediction value
        """

        # For regression, leaf value is the mean of target values
        leaf_value = y.mean()

        # Create and return leaf node
        return Node(value=leaf_value)

    def _create_decision_node(self, feature, threshold):
        """
        Create a decision node.

        Parameters:
        feature : int
            Index of feature to split on
        threshold : float
            Threshold value for the split

        Returns:
        Node
            Decision node
        """

        return Node(
            feature=feature,
            threshold=threshold,
            left=None,
            right=None,
            value=None
        )


    def predict(self, X):
        """
        Predict target values for given input data.

        Parameters:
        X : pandas DataFrame or numpy array (m x n)
            Input feature matrix

        Returns:
        numpy array
            Predicted values
        """

        # Convert Pandas DataFrame to NumPy if needed
        if hasattr(X, "values"):
            X = X.values

        predictions = []

        for x in X:
            pred = self._traverse_tree(x, self.root)
            predictions.append(pred)

        return np.array(predictions)

    def _traverse_tree(self, x, node):
        """
        Traverse the tree for a single data point.

        Parameters:
        x : numpy array (n,)
            One data point
        node : Node
            Current node in the tree

        Returns:
        float
            Predicted value
        """

        # If the node is a leaf, return its prediction value
        if node.value is not None:
            return node.value

        # Otherwise, decide which branch to follow
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        else:
            return self._traverse_tree(x, node.right)

    
    def _weighted_mean_squared_error(self, y_left, y_right):
        """
        Compute weighted mean squared error for a split.

        Parameters:
        y_left : numpy array
            Target values in the left child
        y_right : numpy array
            Target values in the right child

        Returns:
        float
            Weighted MSE of the split
        """

        total_samples = len(y_left) + len(y_right)

        # Weight of each child
        weight_left = len(y_left) / total_samples
        weight_right = len(y_right) / total_samples

        # Variance is the MSE for regression trees
        error = (
            weight_left * y_left.var() +
            weight_right * y_right.var()
        )

        return error
    
    def print_tree(self, node=None, depth=0):
        """
        Print the regression decision tree in a readable format.
        """
        if node is None:
            node = self.root

        indent = "  " * depth

        # Leaf node
        if node.value is not None:
            print(f"{indent}Leaf → Predict: {node.value:.2f}")
            return

        # Decision node
        print(f"{indent}Feature[{node.feature}] <= {node.threshold}")

        print(f"{indent}├─ True:")
        self.print_tree(node.left, depth + 1)

        print(f"{indent}└─ False:")
        self.print_tree(node.right, depth + 1)


    def _get_max_features(self, n_features):
        if self.max_features is None:
            return n_features
        if self.max_features == "sqrt":
            return max(1, int(np.sqrt(n_features)))
        if self.max_features == "log2":
            return max(1, int(np.log2(n_features)))
        if isinstance(self.max_features, int):
            return min(self.max_features, n_features)
        return n_features



In [0]:
import pandas as pd

data = {
    "Area_sqft": [800, 900, 1000, 1100, 1200, 1300, 1400, 1500],
    "Rooms": [2, 2, 3, 3, 3, 4, 4, 4],
    "Distance_km": [18, 17, 15, 14, 12, 10, 8, 6],
    "Age_years": [20, 18, 15, 12, 10, 8, 6, 4],
    "Price_Lakhs": [40, 45, 50, 55, 60, 65, 70, 75]
}

df = pd.DataFrame(data)
df


In [0]:
X = df.drop(columns=["Price_Lakhs"])
y = df["Price_Lakhs"]


In [0]:
tree = DecisionTreeRegressor(
    max_depth=3,
    min_samples_split=2,
    min_samples_leaf=2
)

tree.fit(X, y)
preds = tree.predict(X)


In [0]:
predictions = tree.predict(X)

print("Actual Prices:     ", y.values)
print("Predicted Prices:  ", predictions)


In [0]:
new_house = pd.DataFrame({
    "Area_sqft": [1250],
    "Rooms": [3],
    "Distance_km": [11],
    "Age_years": [9]
})

prediction = tree.predict(new_house)
print("Predicted price:", prediction[0])


In [0]:
print("Root feature index:", tree.root.feature)
print("Root threshold:", tree.root.threshold)


In [0]:
node = tree.root
while node.value is None:
    node = node.left
print("Leftmost leaf value:", node.value)


In [0]:
tree.print_tree()
